# RDR2 Sentiment Analysis — Data Collection

Pulls all raw data used in the project from five sources: Steam Web API, Steam Reviews, YouTube Data API v3, IGDB (via Twitch OAuth), and Google Trends.

Also builds the three curated reference tables (chapters, missions, characters) used for entity matching in the sentiment analysis notebook.

**Run order:** this notebook should run before `02_sentiment_analysis.ipynb`, since that notebook depends on the CSVs produced here.

**Outputs produced by this notebook:**
- `steam_reviews_recent.csv`, `steam_reviews_historical.csv`
- `reference_chapters.csv`, `reference_missions.csv`, `reference_characters.csv`
- `youtube_videos_raw.csv`, `youtube_video_entity_map.csv`, `youtube_comments_raw.csv`
- `igdb_characters_raw.csv`
- `comparison_rdr2_gtav.csv`, `comparison_trends.csv`


**Note on file paths:** this notebook lives in `notebooks/`, so paths point one level up (`../data/...`). The raw Steam reviews and YouTube comments (`data/raw/`) are **not included in the repository**, because they contain other users' review texts and identifiers; rerun `01_data_collection.ipynb` with your own API keys to recreate them.

## Setup
Load API credentials from `.env` — never hardcoded here. See `00_api_connectivity_test.ipynb` to verify credentials before running this notebook.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

STEAM_API_KEY = os.getenv("STEAM_API_KEY")
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")
CLIENT_ID = os.getenv("IGDB_CLIENT_ID")
CLIENT_SECRET = os.getenv("IGDB_CLIENT_SECRET")

print("YOUTUBE_API_KEY loaded:", YOUTUBE_API_KEY is not None)  # boolean check only, never print the key itself

In [ ]:
import requests
import pandas as pd
import time

## Reference tables: chapters, missions, characters

These curated lists drive the regex-based entity matching used throughout `02_sentiment_analysis.ipynb`. Built once here and saved to CSV, then edited/refined directly in that notebook as matching issues were discovered (e.g. alt-name collisions).

**Note:** `reference_chapters.csv` here reflects the *original* 9-row version (Epilogue split into Part 1/Part 2). It was later revised in the analysis notebook to merge the two Epilogue rows into one, after separate tracking of Part 1/Part 2 returned zero matches. The CSV in the project folder reflects that later, corrected version — this cell is kept for reproducibility of the original pipeline.

In [ ]:
chapters = [
    {"chapter_num": 0, "chapter_name": "Prologue", "chapter_alt_names": "Colter"},
    {"chapter_num": 1, "chapter_name": "Chapter 1", "chapter_alt_names": "Outlaws from the West"},
    {"chapter_num": 2, "chapter_name": "Chapter 2", "chapter_alt_names": "Horseshoe Overlook"},
    {"chapter_num": 3, "chapter_name": "Chapter 3", "chapter_alt_names": "Clemens Point"},
    {"chapter_num": 4, "chapter_name": "Chapter 4", "chapter_alt_names": "Shady Belle, Saint Denis"},
    {"chapter_num": 5, "chapter_name": "Chapter 5", "chapter_alt_names": "Guarma"},
    {"chapter_num": 6, "chapter_name": "Chapter 6", "chapter_alt_names": "Beaver Hollow"},
    {"chapter_num": 7, "chapter_name": "Epilogue Part 1", "chapter_alt_names": "Beecher's Hope, John Marston"},
    {"chapter_num": 8, "chapter_name": "Epilogue Part 2", "chapter_alt_names": "Epilogue 2"},
]

df_chapters = pd.DataFrame(chapters)
df_chapters.to_csv("../data/reference/reference_chapters.csv", index=False)
df_chapters

Curated list of 24 named missions, one per chapter category (opening, comedic, character death, twist, etc.) — chosen to be representative rather than exhaustive, since maximizing regex match rates downstream was the priority.

In [ ]:
missions = [
    {"mission_name": "Enter, Pursued by a Memory", "chapter_num": 1, "chapter_name": "Chapter 1", "category": "Opening", "alt_phrases": "opening mission, prologue ending"},
    {"mission_name": "Advertising, the New American Art", "chapter_num": 4, "chapter_name": "Chapter 4", "category": "Comedic", "alt_phrases": "billboard mission"},
    {"mission_name": "Blessed Are the Peacemakers", "chapter_num": 4, "chapter_name": "Chapter 4", "category": "Twist/Betrayal", "alt_phrases": "bank heist, bank job, Saint Denis bank"},
    {"mission_name": "That's Murfree Country", "chapter_num": 3, "chapter_name": "Chapter 3", "category": "Horror/Tension", "alt_phrases": "Murfree Brood, cave mission"},
    {"mission_name": "Money Lending and Other Sins VI", "chapter_num": 2, "chapter_name": "Chapter 2", "category": "Character Moment", "alt_phrases": "Strauss debt collecting"},
    {"mission_name": "A Fisher of Fish", "chapter_num": 2, "chapter_name": "Chapter 2", "category": "Bonding", "alt_phrases": "fishing with Hosea"},
    {"mission_name": "Preaching Forgiveness as He Went", "chapter_num": 3, "chapter_name": "Chapter 3", "category": "Character Death", "alt_phrases": "Sean funeral, Sean death"},
    {"mission_name": "A Fine Night for it", "chapter_num": 4, "chapter_name": "Chapter 4", "category": "Character Death", "alt_phrases": "Angelo Bronte, Bronte alligator"},
    {"mission_name": "Urban Pleasures", "chapter_num": 4, "chapter_name": "Chapter 4", "category": "Character Moment", "alt_phrases": "Saint Denis date, Mary date"},
    {"mission_name": "Banking, the Old American Art", "chapter_num": 4, "chapter_name": "Chapter 4", "category": "Heist", "alt_phrases": "Saint Denis heist, second bank job"},
    {"mission_name": "Fatherhood and Other Dreams", "chapter_num": 5, "chapter_name": "Chapter 5", "category": "Character Death", "alt_phrases": "Guarma, Hosea Guarma"},
    {"mission_name": "The Widow of Willard's Rest", "chapter_num": 3, "chapter_name": "Chapter 3", "category": "Side/Emotional", "alt_phrases": "widow mission, hanging body"},
    {"mission_name": "Hell Hath No Fury", "chapter_num": 6, "chapter_name": "Chapter 6", "category": "Character Death", "alt_phrases": "Hosea death, Lemoyne raid, prison wagon"},
    {"mission_name": "Angelo Bronte, a Man of Honor", "chapter_num": 4, "chapter_name": "Chapter 4", "category": "Character Intro", "alt_phrases": "Bronte introduction"},
    {"mission_name": "Icarus and Friends", "chapter_num": 6, "chapter_name": "Chapter 6", "category": "Twist/Betrayal", "alt_phrases": "Micah betrayal reveal, camp raid"},
    {"mission_name": "The Sheep and the Goats", "chapter_num": 6, "chapter_name": "Chapter 6", "category": "Character Death", "alt_phrases": "Molly death, Molly shot"},
    {"mission_name": "Our Best Selves", "chapter_num": 6, "chapter_name": "Chapter 6", "category": "Moral Choice", "alt_phrases": "final camp choice, help Sadie or Micah"},
    {"mission_name": "American Venom", "chapter_num": 6, "chapter_name": "Chapter 6", "category": "Twist/Betrayal", "alt_phrases": "Dutch abandons Arthur, oil rig"},
    {"mission_name": "Red Dead Redemption", "chapter_num": 6, "chapter_name": "Chapter 6", "category": "Character Death", "alt_phrases": "Arthur death, Arthur's death, mountain ending, sunrise ending, tuberculosis ending"},
    {"mission_name": "Where the Broken Souls Live", "chapter_num": 7, "chapter_name": "Epilogue Part 1", "category": "Character Moment", "alt_phrases": "Beecher's Hope, ranch building"},
    {"mission_name": "Our Mutual Friend", "chapter_num": 7, "chapter_name": "Epilogue Part 1", "category": "Twist/Betrayal", "alt_phrases": "Micah found, Micah location"},
    {"mission_name": "Red Dead Redemption II", "chapter_num": 8, "chapter_name": "Epilogue Part 2", "category": "Finale/Revenge", "alt_phrases": "final mission, Micah fight, Dutch ending, revenge ending"},
    {"mission_name": "A Short Walk in a Pretty Town", "chapter_num": 2, "chapter_name": "Chapter 2", "category": "Comedic", "alt_phrases": "Valentine bar fight, Micah first meeting"},
    {"mission_name": "Polite Society, Valentine Style", "chapter_num": 1, "chapter_name": "Chapter 1", "category": "Comedic", "alt_phrases": "Valentine brawl"},
]

df_missions = pd.DataFrame(missions)
df_missions.to_csv("../data/reference/reference_missions.csv", index=False)
df_missions

25 named characters spanning protagonists, gang members, love interests, and antagonists — curated rather than exhaustive, same rationale as the missions list.

In [ ]:
characters = [
    {"character_name": "Arthur Morgan", "role": "Protagonist", "alt_names": "Arthur, Morgan"},
    {"character_name": "Dutch van der Linde", "role": "Gang Leader", "alt_names": "Dutch, van der Linde"},
    {"character_name": "John Marston", "role": "Protagonist", "alt_names": "John, Marston"},
    {"character_name": "Hosea Matthews", "role": "Gang Elder", "alt_names": "Hosea"},
    {"character_name": "Sadie Adler", "role": "Gang Member", "alt_names": "Sadie, Sadie Adler"},
    {"character_name": "Charles Smith", "role": "Gang Member", "alt_names": "Charles"},
    {"character_name": "Javier Escuella", "role": "Gang Member", "alt_names": "Javier, Escuella"},
    {"character_name": "Bill Williamson", "role": "Gang Member", "alt_names": "Bill, Williamson"},
    {"character_name": "Micah Bell", "role": "Antagonist", "alt_names": "Micah, Micah Bell"},
    {"character_name": "Lenny Summers", "role": "Gang Member", "alt_names": "Lenny"},
    {"character_name": "Karen Jones", "role": "Gang Member", "alt_names": "Karen"},
    {"character_name": "Uncle", "role": "Gang Member", "alt_names": "Uncle"},
    {"character_name": "Pearson", "role": "Gang Member", "alt_names": "Pearson, Mr. Pearson"},
    {"character_name": "Kieran Duffy", "role": "Gang Member", "alt_names": "Kieran"},
    {"character_name": "Tilly Jackson", "role": "Gang Member", "alt_names": "Tilly"},
    {"character_name": "Mary-Beth Gaskill", "role": "Gang Member", "alt_names": "Mary-Beth, Mary Beth"},
    {"character_name": "Abigail Roberts", "role": "Gang Member", "alt_names": "Abigail"},
    {"character_name": "Jack Marston", "role": "Gang Member", "alt_names": "Jack"},
    {"character_name": "Molly O'Shea", "role": "Gang Member", "alt_names": "Molly, Molly O'Shea"},
    {"character_name": "Mary Linton", "role": "Love Interest", "alt_names": "Mary Linton, Mary Gillis"},
    {"character_name": "Colm O'Driscoll", "role": "Antagonist", "alt_names": "Colm, O'Driscoll"},
    {"character_name": "Leviticus Cornwall", "role": "Antagonist", "alt_names": "Cornwall, Leviticus"},
    {"character_name": "Angelo Bronte", "role": "Antagonist", "alt_names": "Bronte, Angelo Bronte"},
    {"character_name": "Agent Milton", "role": "Antagonist", "alt_names": "Milton, Agent Milton"},
    {"character_name": "Agent Ross", "role": "Antagonist", "alt_names": "Ross, Agent Ross"},
]

df_characters = pd.DataFrame(characters)
df_characters.to_csv("../data/reference/reference_characters.csv", index=False)
df_characters

## Steam reviews — recent pull

Pulls the most recent reviews (`filter="recent"`) for both games. This is the primary corpus for the character/chapter/mission sentiment analysis, since recent reviews are more likely to reference specific story content in detail.

In [ ]:
def fetch_steam_reviews(appid, game_name, max_reviews=2000, language="english"):
    """
    Pulls reviews for a given Steam appid using cursor-based pagination.
    Stops when max_reviews is hit, or when the cursor stops advancing (no more reviews).
    """
    reviews_data = []
    cursor = "*"
    seen_cursors = set()

    url = f"https://store.steampowered.com/appreviews/{appid}"

    while len(reviews_data) < max_reviews:
        params = {
            "json": 1,
            "filter": "recent",
            "language": language,
            "num_per_page": 100,
            "cursor": cursor
        }
        r = requests.get(url, params=params)
        data = r.json()

        reviews = data.get("reviews", [])
        if not reviews:
            print(f"No more reviews returned for {game_name}. Stopping.")
            break

        for rev in reviews:
            reviews_data.append({
                "game": game_name,
                "appid": appid,
                "review_id": rev.get("recommendationid"),
                "review_text": rev.get("review"),
                "timestamp_created": rev.get("timestamp_created"),
                "timestamp_updated": rev.get("timestamp_updated"),
                "voted_up": rev.get("voted_up"),
                "votes_up": rev.get("votes_up"),
                "votes_funny": rev.get("votes_funny"),
                "playtime_forever_minutes": rev.get("author", {}).get("playtime_forever"),
                "playtime_at_review_minutes": rev.get("author", {}).get("playtime_at_review"),
                "num_games_owned": rev.get("author", {}).get("num_games_owned"),
                "num_reviews_by_author": rev.get("author", {}).get("num_reviews"),
            })

        new_cursor = data.get("cursor")
        if new_cursor is None or new_cursor in seen_cursors:
            print(f"Cursor stopped advancing for {game_name}. Reached end of available reviews.")
            break
        seen_cursors.add(new_cursor)
        cursor = new_cursor

        print(f"{game_name}: pulled {len(reviews_data)} reviews so far...")
        time.sleep(1)  # be polite to the API, avoid rate limiting

    return pd.DataFrame(reviews_data[:max_reviews])


# Pull for both games
rdr2_reviews = fetch_steam_reviews(appid=1174180, game_name="RDR2", max_reviews=2000)
gtav_reviews = fetch_steam_reviews(appid=271590, game_name="GTA V", max_reviews=2000)

# Combine into a single raw file (game column lets you split/filter later in Power BI)
all_reviews = pd.concat([rdr2_reviews, gtav_reviews], ignore_index=True)

all_reviews.to_csv("../data/raw/steam_reviews_recent.csv", index=False)
print(f"Saved {len(all_reviews)} total reviews to steam_reviews_recent.csv")

Quick sanity check: shape, per-game split, and null counts before moving on.

In [ ]:
df = pd.read_csv("../data/raw/steam_reviews_recent.csv")

print(df.shape)
print(df["game"].value_counts())
print(df.isnull().sum())
df.head(3)

## Steam reviews — historical pull

Pulls a broader spread of reviews across the game's full history (`filter="all"`, sorted by helpfulness, unrestricted `day_range`), needed specifically for the sentiment-over-time analysis (legacy inflation, epilogue effect). The "recent" pull above is time-skewed toward now and isn't suitable for that.

In [ ]:
def fetch_steam_reviews_historical(appid, game_name, max_reviews=3000, language="english"):
    """
    Pulls reviews using filter='all' (sorted by helpfulness by default),
    which gives a broader spread across the game's review history —
    needed for sentiment-over-time analysis (legacy inflation, epilogue effect, patch reactions).
    """
    reviews_data = []
    cursor = "*"
    seen_cursors = set()

    url = f"https://store.steampowered.com/appreviews/{appid}"

    while len(reviews_data) < max_reviews:
        params = {
            "json": 1,
            "filter": "all",
            "language": language,
            "num_per_page": 100,
            "cursor": cursor,
            "day_range": 9223372036854775807  # max range, ensures full history is eligible
        }
        r = requests.get(url, params=params)
        data = r.json()

        reviews = data.get("reviews", [])
        if not reviews:
            print(f"No more reviews returned for {game_name}. Stopping.")
            break

        for rev in reviews:
            reviews_data.append({
                "game": game_name,
                "appid": appid,
                "review_id": rev.get("recommendationid"),
                "review_text": rev.get("review"),
                "timestamp_created": rev.get("timestamp_created"),
                "timestamp_updated": rev.get("timestamp_updated"),
                "voted_up": rev.get("voted_up"),
                "votes_up": rev.get("votes_up"),
                "votes_funny": rev.get("votes_funny"),
                "playtime_forever_minutes": rev.get("author", {}).get("playtime_forever"),
                "playtime_at_review_minutes": rev.get("author", {}).get("playtime_at_review"),
                "num_games_owned": rev.get("author", {}).get("num_games_owned"),
                "num_reviews_by_author": rev.get("author", {}).get("num_reviews"),
            })

        new_cursor = data.get("cursor")
        if new_cursor is None or new_cursor in seen_cursors:
            print(f"Cursor stopped advancing for {game_name}. Reached end of available reviews.")
            break
        seen_cursors.add(new_cursor)
        cursor = new_cursor

        print(f"{game_name}: pulled {len(reviews_data)} historical reviews so far...")
        time.sleep(1)

    return pd.DataFrame(reviews_data[:max_reviews])


rdr2_historical = fetch_steam_reviews_historical(appid=1174180, game_name="RDR2", max_reviews=3000)
gtav_historical = fetch_steam_reviews_historical(appid=271590, game_name="GTA V", max_reviews=3000)

all_historical = pd.concat([rdr2_historical, gtav_historical], ignore_index=True)
all_historical.to_csv("../data/raw/steam_reviews_historical.csv", index=False)
print(f"Saved {len(all_historical)} total historical reviews to steam_reviews_historical.csv")

In [ ]:
df = pd.read_csv("../data/raw/steam_reviews_historical.csv")

print(df.shape)
print(df["game"].value_counts())
print(df.isnull().sum())
df.head(3)

Check the actual date range and per-year distribution the historical pull achieved — this determines what years the sentiment-over-time analysis can meaningfully cover.

In [ ]:
df_hist = pd.read_csv("../data/raw/steam_reviews_historical.csv")
df_hist["date_created"] = pd.to_datetime(df_hist["timestamp_created"], unit="s")

print(df_hist["date_created"].min(), "to", df_hist["date_created"].max())
print(df_hist.groupby("game")["date_created"].agg(["min", "max"]))

# Distribution by year, per game
df_hist["year"] = df_hist["date_created"].dt.year
print(df_hist.groupby(["game", "year"]).size())

## YouTube data collection

Three-stage pull: (1) search for videos per mission/character, (2) fetch view/like/comment stats for each unique video found, (3) fetch top comments per video. Each function notes its YouTube API quota cost, since the daily quota is limited.

In [ ]:
# Load reference lists
df_missions = pd.read_csv("../data/reference/reference_missions.csv")
df_characters = pd.read_csv("../data/reference/reference_characters.csv")

def youtube_search(query, max_results=3):
    """
    Searches YouTube for a query, returns basic video metadata.
    Cost: 100 quota units per call.
    """
    url = "https://www.googleapis.com/youtube/v3/search"
    params = {
        "part": "snippet",
        "q": query,
        "type": "video",
        "maxResults": max_results,
        "key": YOUTUBE_API_KEY
    }
    r = requests.get(url, params=params)
    if r.status_code != 200:
        print(f"Search failed for '{query}': {r.status_code} {r.text[:200]}")
        return []

    results = []
    for item in r.json().get("items", []):
        results.append({
            "search_query": query,
            "video_id": item["id"]["videoId"],
            "title": item["snippet"]["title"],
            "published_at": item["snippet"]["publishedAt"],
            "channel_title": item["snippet"]["channelTitle"]
        })
    return results


def youtube_video_stats(video_ids):
    """
    Pulls view/like/comment counts for a batch of video IDs (up to 50 per call).
    Cost: 1 quota unit per call.
    """
    url = "https://www.googleapis.com/youtube/v3/videos"
    params = {
        "part": "statistics",
        "id": ",".join(video_ids),
        "key": YOUTUBE_API_KEY
    }
    r = requests.get(url, params=params)
    if r.status_code != 200:
        print(f"Stats fetch failed: {r.status_code} {r.text[:200]}")
        return {}

    stats = {}
    for item in r.json().get("items", []):
        stats[item["id"]] = item.get("statistics", {})
    return stats


def youtube_comments(video_id, max_comments=30):
    """
    Pulls top-level comments for a video.
    Cost: 1 quota unit per call (per page of up to 100 comments).
    """
    url = "https://www.googleapis.com/youtube/v3/commentThreads"
    params = {
        "part": "snippet",
        "videoId": video_id,
        "maxResults": min(max_comments, 100),
        "order": "relevance",
        "textFormat": "plainText",
        "key": YOUTUBE_API_KEY
    }
    r = requests.get(url, params=params)
    if r.status_code != 200:
        # Comments can be disabled on some videos - not a pipeline error, just skip
        return []

    comments = []
    for item in r.json().get("items", []):
        top = item["snippet"]["topLevelComment"]["snippet"]
        comments.append({
            "video_id": video_id,
            "comment_text": top.get("textDisplay"),
            "like_count": top.get("likeCount"),
            "published_at": top.get("publishedAt"),
            "author": top.get("authorDisplayName")
        })
    return comments


# Build combined list of search targets: missions + characters
search_targets = []

for _, row in df_missions.iterrows():
    search_targets.append({"entity_type": "mission", "entity_name": row["mission_name"], "query": f"RDR2 {row['mission_name']}"})

for _, row in df_characters.iterrows():
    search_targets.append({"entity_type": "character", "entity_name": row["character_name"], "query": f"RDR2 {row['character_name']}"})

print(f"Total search queries to run: {len(search_targets)}")

# Stage 1: Search
all_videos = []

for target in search_targets:
    results = youtube_search(target["query"], max_results=3)
    for r in results:
        r["entity_type"] = target["entity_type"]
        r["entity_name"] = target["entity_name"]
        all_videos.append(r)
    print(f"Searched: {target['query']} -> {len(results)} videos")
    time.sleep(0.5)

df_videos = pd.DataFrame(all_videos)
df_videos.to_csv("../data/processed/youtube_videos_raw.csv", index=False)
print(f"\nSaved {len(df_videos)} videos to youtube_videos_raw.csv")

In [ ]:
df_videos = pd.read_csv("../data/processed/youtube_videos_raw.csv")
print(df_videos.shape)
print(df_videos["entity_type"].value_counts())
print("Duplicate video_ids:", df_videos["video_id"].duplicated().sum())
df_videos.head(5)

A video can surface for multiple search queries (e.g. a video about Arthur might also match a mission search), so we split into a deduped video table plus a separate many-to-many entity map — keeps the relationship without duplicating video metadata.

In [ ]:
# Create the video-to-entity join table (preserves all entity/video relationships, including duplicates)
df_video_entity_map = df_videos[["video_id", "entity_type", "entity_name", "search_query"]].copy()
df_video_entity_map.to_csv("../data/processed/youtube_video_entity_map.csv", index=False)

# Create the deduped video metadata table (one row per unique video)
df_videos_deduped = df_videos.drop_duplicates(subset="video_id", keep="first").drop(columns=["entity_type", "entity_name", "search_query"])
df_videos_deduped.to_csv("../data/processed/youtube_videos_raw.csv", index=False)  # Overwrite with deduped version

print(f"Unique videos: {len(df_videos_deduped)}")
print(f"Video-entity mappings: {len(df_video_entity_map)}")

In [ ]:
def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

all_stats = {}
video_ids = df_videos_deduped["video_id"].tolist()

for batch in chunk_list(video_ids, 50):  # API allows up to 50 IDs per call
    stats = youtube_video_stats(batch)
    all_stats.update(stats)
    time.sleep(0.5)

# Merge stats into the deduped video dataframe
df_videos_deduped["view_count"] = df_videos_deduped["video_id"].map(lambda v: all_stats.get(v, {}).get("viewCount"))
df_videos_deduped["like_count"] = df_videos_deduped["video_id"].map(lambda v: all_stats.get(v, {}).get("likeCount"))
df_videos_deduped["comment_count"] = df_videos_deduped["video_id"].map(lambda v: all_stats.get(v, {}).get("commentCount"))

df_videos_deduped.to_csv("../data/processed/youtube_videos_raw.csv", index=False)
print("Stats merged and saved.")
df_videos_deduped.head(5)

In [ ]:
all_comments = []

for i, video_id in enumerate(df_videos_deduped["video_id"], 1):
    comments = youtube_comments(video_id, max_comments=30)
    all_comments.extend(comments)
    if i % 20 == 0:
        print(f"Processed {i}/{len(df_videos_deduped)} videos, {len(all_comments)} comments so far...")
    time.sleep(0.3)

df_comments = pd.DataFrame(all_comments)
df_comments.to_csv("../data/raw/youtube_comments_raw.csv", index=False)
print(f"\nSaved {len(df_comments)} comments to youtube_comments_raw.csv")

In [ ]:
df_comments.head()

In [ ]:
print("Total comments:", len(df_comments))
print("Videos with zero comments:", df_videos_deduped["video_id"].isin(df_comments["video_id"]).value_counts())
print("Avg comments per video:", df_comments.groupby("video_id").size().mean())
df_comments["comment_text"].str.len().describe()

## IGDB character enrichment

Looks up each character on IGDB (via Twitch OAuth) to pull an official description — used for potential tooltip/context content. Not all characters (especially minor gang members) have IGDB entries.

In [ ]:
def get_igdb_token():
    """Exchange IGDB/Twitch client credentials for a short-lived OAuth access token."""
    url = "https://id.twitch.tv/oauth2/token"
    params = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials"
    }
    r = requests.post(url, params=params)
    return r.json()["access_token"]

token = get_igdb_token()

def igdb_character_lookup(character_name):
    """Search IGDB for a character by name, return the top match (or None if not found)."""
    url = "https://api.igdb.com/v4/characters"
    headers = {
        "Client-ID": CLIENT_ID,
        "Authorization": f"Bearer {token}"
    }
    body = f'search "{character_name}"; fields name,games,description; limit 1;'
    r = requests.post(url, headers=headers, data=body)
    if r.status_code != 200:
        print(f"Failed for {character_name}: {r.status_code}")
        return None
    results = r.json()
    return results[0] if results else None


df_characters = pd.read_csv("../data/reference/reference_characters.csv")

igdb_results = []
for _, row in df_characters.iterrows():
    result = igdb_character_lookup(row["character_name"])
    igdb_results.append({
        "character_name": row["character_name"],
        "igdb_id": result.get("id") if result else None,
        "igdb_name": result.get("name") if result else None,
        "igdb_description": result.get("description") if result else None,
        "found_in_igdb": result is not None
    })
    time.sleep(0.3)

df_igdb = pd.DataFrame(igdb_results)
df_igdb.to_csv("../data/processed/igdb_characters_raw.csv", index=False)
print(f"Found {df_igdb['found_in_igdb'].sum()} / {len(df_igdb)} characters in IGDB")
df_igdb

## Comparison metrics: RDR2 vs. GTA V

Pulls a point-in-time snapshot (current player count, SteamSpy owner/pricing estimates, review score summary) for both games — used for the Q4 comparative analysis.

In [ ]:
from datetime import datetime

def get_steam_current_players(appid):
    """Fetch the live current player count for a Steam appid."""
    url = f"https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={appid}"
    r = requests.get(url)
    return r.json().get("response", {}).get("player_count")

def get_steamspy_data(appid):
    """Fetch SteamSpy's estimated owners, pricing, and playtime data for an appid."""
    url = f"https://steamspy.com/api.php?request=appdetails&appid={appid}"
    r = requests.get(url)
    return r.json()

def get_steam_review_summary(appid):
    """Fetch Steam's own aggregate review summary (score description, totals)."""
    url = f"https://store.steampowered.com/appreviews/{appid}?json=1&num_per_page=0"
    r = requests.get(url)
    return r.json().get("query_summary", {})


games = {"RDR2": 1174180, "GTA V": 271590}
comparison_rows = []

for game_name, appid in games.items():
    player_count = get_steam_current_players(appid)
    time.sleep(0.5)
    steamspy = get_steamspy_data(appid)
    time.sleep(0.5)
    review_summary = get_steam_review_summary(appid)
    time.sleep(0.5)

    comparison_rows.append({
        "game": game_name,
        "appid": appid,
        "pulled_at": datetime.now().isoformat(),
        "current_players": player_count,
        "owners_estimate": steamspy.get("owners"),
        "price": steamspy.get("price"),
        "positive_reviews": steamspy.get("positive"),
        "negative_reviews": steamspy.get("negative"),
        "average_playtime_forever_min": steamspy.get("average_forever"),
        "median_playtime_forever_min": steamspy.get("median_forever"),
        "review_score_desc": review_summary.get("review_score_desc"),
        "total_reviews": review_summary.get("total_reviews"),
        "total_positive": review_summary.get("total_positive"),
        "total_negative": review_summary.get("total_negative"),
    })

df_comparison = pd.DataFrame(comparison_rows)
df_comparison.to_csv("../data/processed/comparison_rdr2_gtav.csv", index=False)
df_comparison

## Google Trends: search interest over time

Pulls relative search interest (0-100 index) for both games since RDR2's 2019 PC launch — used as a proxy for ongoing cultural attention (Q4).

In [ ]:
from pytrends.request import TrendReq

pytrends = TrendReq(hl="en-US", tz=360)
pytrends.build_payload(["Red Dead Redemption 2", "GTA V"], timeframe="2019-01-01 2026-07-01")
df_trends = pytrends.interest_over_time()
df_trends = df_trends.drop(columns=["isPartial"], errors="ignore")
df_trends.to_csv("../data/processed/comparison_trends.csv")
print(df_trends.head())